# Silver Layer - Suppliers

Transform raw Bronze data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.suppliers`  
**Target:** `end-to-end_pipeline.silver.suppliers`

**Approach:** Profile → Inspect → Transform → Validate


## Step 1: Profile Bronze Data

**Inspect data quality issues before transformation:**

* Duplicate supplier_ids
* NULL values in key fields (supplier_id, supplier_name, country, supplier_rating, active_status)
* Inconsistent country values (" germany " with spaces)
* Invalid supplier ratings (outside 1-5 range)

This single query checks all quality dimensions.

In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE BRONZE SUPPLIERS
-- Purpose: Identify duplicates, nulls, invalid values,
--          and categorical inconsistencies
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT supplier_id) AS distinct_supplier_ids,
    COUNT(*) - COUNT(DISTINCT supplier_id) AS duplicate_suppliers,

    SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END)
        AS null_supplier_ids,

    SUM(CASE WHEN supplier_name IS NULL THEN 1 ELSE 0 END)
        AS null_supplier_names,

    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END)
        AS null_countries,

    SUM(CASE WHEN supplier_rating IS NULL THEN 1 ELSE 0 END)
        AS null_supplier_ratings,

    SUM(CASE
        WHEN supplier_rating IS NOT NULL
             AND (supplier_rating < 1 OR supplier_rating > 5)
        THEN 1 ELSE 0
    END) AS invalid_supplier_ratings,

    SUM(CASE WHEN active_status IS NULL THEN 1 ELSE 0 END)
        AS null_active_status,

    SUM(CASE
        WHEN supplier_id != TRIM(supplier_id) THEN 1
        ELSE 0
    END) AS supplier_id_spaces,

    COUNT(DISTINCT country) AS country_variations

FROM `end-to-end_pipeline`.bronze.suppliers;

## Step 2: Inspect Categorical Values

**Review actual country and rating values to identify standardization needs:**

* Country variations (" germany " with spaces → Germany)
* Supplier rating outliers (7.5 is outside valid 1-5 range)
* NULL ratings (acceptable - preserved as NULL)

This inspection guides the explicit CASE logic in the transformation.

In [0]:
%sql

-- ============================================================
-- CELL 2A: INSPECT COUNTRY VALUES
-- ============================================================

SELECT
    country,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.suppliers
GROUP BY country
ORDER BY records DESC;

In [0]:
%sql

-- ============================================================
-- CELL 2B: INSPECT SUPPLIER RATINGS
-- ============================================================

SELECT
    supplier_rating,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.suppliers
GROUP BY supplier_rating
ORDER BY supplier_rating;

## Step 3: Transform to Silver

**Apply all data quality fixes in one pass:**

**Data Cleaning:**
* TRIM whitespace from all identifiers and text (handles " germany " → "germany")
* Standardize names, countries, categories with INITCAP

**Categorical Standardization:**
* Standardize country casing (germany → Germany)
* Standardize supplier_category casing
* Standardize active_status casing

**Data Type Enforcement:**
* Cast supplier_rating to DECIMAL(3,1)
* Convert invalid ratings (7.5) → NULL
* Preserve NULL ratings as NULL
* Convert contract_start_date to DATE format

**Deduplication:**
* ROW_NUMBER() keeps first occurrence per supplier_id

This creates a clean, analytics-ready Silver table.

In [0]:
%sql

-- ============================================================
-- CELL 3: TRANSFORM BRONZE → SILVER SUPPLIERS
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.suppliers AS

WITH cleaned AS (

    SELECT
        TRIM(supplier_id) AS supplier_id,

        INITCAP(TRIM(supplier_name)) AS supplier_name,

        INITCAP(TRIM(country)) AS country,

        INITCAP(TRIM(supplier_category)) AS supplier_category,

        CASE
            WHEN supplier_rating BETWEEN 1 AND 5
                THEN CAST(supplier_rating AS DECIMAL(3,1))
            ELSE NULL
        END AS supplier_rating,

        CAST(lead_time_days AS INT) AS lead_time_days,

        TRY_TO_DATE(
            contract_start_date,
            'yyyy-MM-dd'
        ) AS contract_start_date,

        INITCAP(TRIM(active_status)) AS active_status,

        ROW_NUMBER() OVER (
            PARTITION BY TRIM(supplier_id)
            ORDER BY supplier_id
        ) AS row_num

    FROM `end-to-end_pipeline`.bronze.suppliers
)

SELECT
    supplier_id,
    supplier_name,
    country,
    supplier_category,
    supplier_rating,
    lead_time_days,
    contract_start_date,
    active_status

FROM cleaned

WHERE row_num = 1
  AND supplier_id IS NOT NULL;

## Step 4: Validate Silver Data

**Verify all transformations were successful.**

**Expected Results:**
* total_rows = 40 (removed 1 duplicate from 41)
* distinct_supplier_ids = 40
* remaining_duplicates = 0
* null_supplier_ids = 0
* null_supplier_ratings = 1 (preserved NULL + converted 7.5 → NULL = 2 total)
* invalid_supplier_ratings = 0 (all ratings within 1-5 range)
* invalid_lead_times = 0
* invalid_contract_dates = 0
* country_spacing_issues = 0
* validation_status = 'PASS'

If any metric is unexpected, the transformation has an issue.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE SILVER SUPPLIERS
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT supplier_id) AS distinct_supplier_ids,
        COUNT(*) - COUNT(DISTINCT supplier_id) AS remaining_duplicates,

        SUM(CASE
            WHEN supplier_id IS NULL THEN 1
            ELSE 0
        END) AS null_supplier_ids,

        SUM(CASE
            WHEN supplier_rating IS NULL THEN 1
            ELSE 0
        END) AS null_supplier_ratings,

        SUM(CASE
            WHEN supplier_rating < 1 OR supplier_rating > 5
            THEN 1
            ELSE 0
        END) AS invalid_supplier_ratings,

        SUM(CASE
            WHEN lead_time_days < 0 THEN 1
            ELSE 0
        END) AS invalid_lead_times,

        SUM(CASE
            WHEN contract_start_date IS NULL THEN 1
            ELSE 0
        END) AS invalid_contract_dates,

        SUM(CASE
            WHEN country != TRIM(country) THEN 1
            ELSE 0
        END) AS country_spacing_issues,

        SUM(CASE
            WHEN active_status IN ('Active', 'Inactive')
            THEN 0
            ELSE 1
        END) AS invalid_active_status

    FROM `end-to-end_pipeline`.silver.suppliers
)

SELECT
    *,

    CASE
        WHEN total_rows = 40
            AND distinct_supplier_ids = 40
            AND remaining_duplicates = 0
            AND null_supplier_ids = 0
            AND null_supplier_ratings = 2
            AND invalid_supplier_ratings = 0
            AND invalid_lead_times = 0
            AND invalid_contract_dates = 0
            AND country_spacing_issues = 0
            AND invalid_active_status = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation;